In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)
import tqdm as notebook_tqdm
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.13.0+cpu
CUDA available: False


In [4]:
GRAPH_PATH = Path("../data/processed/edge_graph.pt")

edge_graph = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

edge_index = edge_graph["edge_index"]
edge_features = edge_graph["edge_attr"]
edge_labels = edge_graph["edge_label"]

print("edge_index:", edge_index.shape)
print("edge_features:", edge_features.shape)
print("edge_labels:", edge_labels.shape)

edge_index: torch.Size([2, 286095])
edge_features: torch.Size([286095, 67])
edge_labels: torch.Size([286095])


In [5]:
print("Benign flows:", (edge_labels == 0).sum().item())
print("Attack flows:", (edge_labels == 1).sum().item())

print(
    "Attack percentage:",
    round((edge_labels == 1).float().mean().item() * 100, 2),
    "%"
)

Benign flows: 127291
Attack flows: 158804
Attack percentage: 55.51 %


In [6]:
num_edges = edge_labels.shape[0]

all_edge_indices = np.arange(num_edges)
labels_numpy = edge_labels.numpy()

train_indices, temp_indices = train_test_split(
    all_edge_indices,
    test_size=0.30,
    random_state=42,
    stratify=labels_numpy
)

val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=labels_numpy[temp_indices]
)

train_indices = torch.tensor(train_indices, dtype=torch.long)
val_indices = torch.tensor(val_indices, dtype=torch.long)
test_indices = torch.tensor(test_indices, dtype=torch.long)

print("Training edges:", len(train_indices))
print("Validation edges:", len(val_indices))
print("Testing edges:", len(test_indices))

Training edges: 200266
Validation edges: 42914
Testing edges: 42915


In [7]:
def show_split_distribution(name, indices):
    split_labels = edge_labels[indices]

    print(f"\n{name}")
    print("Total:", len(indices))
    print("Benign:", (split_labels == 0).sum().item())
    print("Attack:", (split_labels == 1).sum().item())


show_split_distribution("Training", train_indices)
show_split_distribution("Validation", val_indices)
show_split_distribution("Testing", test_indices)


Training
Total: 200266
Benign: 89103
Attack: 111163

Validation
Total: 42914
Benign: 19094
Attack: 23820

Testing
Total: 42915
Benign: 19094
Attack: 23821


In [8]:
num_nodes = int(edge_index.max().item()) + 1

NODE_EMBEDDING_SIZE = 16

node_embeddings = nn.Embedding(
    num_nodes,
    NODE_EMBEDDING_SIZE
)

print("Number of nodes:", num_nodes)
print("Node embedding size:", NODE_EMBEDDING_SIZE)

Number of nodes: 4404
Node embedding size: 16


Node embeddings
      ↓
GCN layer
      ↓
Source node embedding + destination node embedding
      ↓
67 edge features
      ↓
Classifier
      ↓
Benign / Attack

In [9]:
class EdgeGCN(nn.Module):
    def __init__(
        self,
        num_nodes,
        edge_feature_size,
        embedding_size=16,
        hidden_size=32
    ):
        super().__init__()

        self.node_embeddings = nn.Embedding(
            num_nodes,
            embedding_size
        )

        self.conv1 = GCNConv(
            embedding_size,
            hidden_size
        )

        self.conv2 = GCNConv(
            hidden_size,
            hidden_size
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_size * 2 + edge_feature_size,
                64
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, edge_index, edge_features):
        node_ids = torch.arange(
            self.node_embeddings.num_embeddings,
            device=edge_index.device
        )

        x = self.node_embeddings(node_ids)

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        source_nodes = edge_index[0]
        destination_nodes = edge_index[1]

        source_embeddings = x[source_nodes]
        destination_embeddings = x[destination_nodes]

        combined_features = torch.cat(
            [
                source_embeddings,
                destination_embeddings,
                edge_features
            ],
            dim=1
        )

        output = self.classifier(combined_features)

        return output

In [10]:
device = torch.device("cpu")

model = EdgeGCN(
    num_nodes=num_nodes,
    edge_feature_size=edge_features.shape[1],
    embedding_size=16,
    hidden_size=32
).to(device)

print(model)

EdgeGCN(
  (node_embeddings): Embedding(4404, 16)
  (conv1): GCNConv(16, 32)
  (conv2): GCNConv(32, 32)
  (classifier): Sequential(
    (0): Linear(in_features=131, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [11]:
train_labels = edge_labels[train_indices]

class_counts = torch.bincount(train_labels)

class_weights = class_counts.sum() / (
    len(class_counts) * class_counts.float()
)

class_weights = class_weights.to(device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

Class counts: tensor([ 89103, 111163])
Class weights: tensor([1.1238, 0.9008])


In [12]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

print("Loss function and optimizer created.")

Loss function and optimizer created.


In [13]:
edge_index = edge_index.to(device)
edge_features = edge_features.to(device)
edge_labels = edge_labels.to(device)

train_indices = train_indices.to(device)
val_indices = val_indices.to(device)
test_indices = test_indices.to(device)

print("All tensors moved to:", device)

All tensors moved to: cpu


In [14]:
def evaluate_model(indices):
    model.eval()

    with torch.no_grad():
        logits = model(
            edge_index,
            edge_features
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        predictions = logits.argmax(dim=1)

        y_true = edge_labels[indices].cpu().numpy()
        y_pred = predictions[indices].cpu().numpy()
        y_prob = probabilities[indices].cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )
    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )
    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )
    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )
    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

In [15]:
EPOCHS = 30

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    optimizer.zero_grad()

    logits = model(
        edge_index,
        edge_features
    )

    loss = criterion(
        logits[train_indices],
        edge_labels[train_indices]
    )

    loss.backward()
    optimizer.step()

    train_metrics = evaluate_model(train_indices)
    val_metrics = evaluate_model(val_indices)

    history.append({
        "epoch": epoch,
        "loss": loss.item(),
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_pr_auc": val_metrics["pr_auc"]
    })

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {loss.item():.4f} | "
            f"Train F1: {train_metrics['f1']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f} | "
            f"Val PR-AUC: {val_metrics['pr_auc']:.4f}"
        )

Epoch 01 | Loss: 0.6642 | Train F1: 0.9144 | Val F1: 0.9155 | Val PR-AUC: 0.9417
Epoch 05 | Loss: 0.5257 | Train F1: 0.9903 | Val F1: 0.9909 | Val PR-AUC: 0.9991
Epoch 10 | Loss: 0.3868 | Train F1: 0.9921 | Val F1: 0.9926 | Val PR-AUC: 0.9993
Epoch 15 | Loss: 0.2596 | Train F1: 0.9934 | Val F1: 0.9938 | Val PR-AUC: 0.9993
Epoch 20 | Loss: 0.1520 | Train F1: 0.9953 | Val F1: 0.9956 | Val PR-AUC: 0.9994
Epoch 25 | Loss: 0.0786 | Train F1: 0.9975 | Val F1: 0.9974 | Val PR-AUC: 0.9995
Epoch 30 | Loss: 0.0400 | Train F1: 0.9978 | Val F1: 0.9977 | Val PR-AUC: 0.9997


In [16]:
history_df = pd.DataFrame(history)

HISTORY_PATH = Path("../outputs/day9_training_history.csv")

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print("Saved:", HISTORY_PATH)

Saved: ..\outputs\day9_training_history.csv


In [17]:
test_metrics = evaluate_model(test_indices)

print("TEST RESULTS")
print("=" * 40)

for metric_name, metric_value in test_metrics.items():
    print(
        f"{metric_name}: {metric_value:.4f}"
    )

TEST RESULTS
accuracy: 0.9980
precision: 0.9980
recall: 0.9984
f1: 0.9982
roc_auc: 0.9995
pr_auc: 0.9998


In [18]:
model.eval()

with torch.no_grad():
    test_logits = model(
        edge_index,
        edge_features
    )

test_predictions = test_logits.argmax(dim=1)

y_true = edge_labels[test_indices].cpu().numpy()
y_pred = test_predictions[test_indices].cpu().numpy()

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Benign", "Attack"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     19094
      Attack       1.00      1.00      1.00     23821

    accuracy                           1.00     42915
   macro avg       1.00      1.00      1.00     42915
weighted avg       1.00      1.00      1.00     42915



In [19]:
cm = confusion_matrix(
    y_true,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Benign", "Actual Attack"],
    columns=["Predicted Benign", "Predicted Attack"]
)

cm_df

,Predicted Benign,Predicted Attack
Actual Benign,19047,47
Actual Attack,39,23782


In [20]:
CM_PATH = Path("../outputs/day9_confusion_matrix.csv")

cm_df.to_csv(CM_PATH)

print("Saved:", CM_PATH)

Saved: ..\outputs\day9_confusion_matrix.csv


In [21]:
MODEL_PATH = Path("../outputs/edge_gcn_model.pt")

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_nodes": num_nodes,
        "edge_feature_size": edge_features.shape[1],
        "embedding_size": 16,
        "hidden_size": 32,
        "class_weights": class_weights.cpu()
    },
    MODEL_PATH
)

print("Saved:", MODEL_PATH)

Saved: ..\outputs\edge_gcn_model.pt


In [22]:
print("=" * 50)
print("DAY 9 COMPLETED")
print("=" * 50)

print("Model: Edge-level GCN")
print("Nodes:", num_nodes)
print("Edges:", edge_index.shape[1])
print("Edge features:", edge_features.shape[1])

print("\nTest metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nFiles created:")
print("1.", HISTORY_PATH)
print("2.", CM_PATH)
print("3.", MODEL_PATH)

DAY 9 COMPLETED
Model: Edge-level GCN
Nodes: 4404
Edges: 286095
Edge features: 67

Test metrics:
accuracy: 0.9980
precision: 0.9980
recall: 0.9984
f1: 0.9982
roc_auc: 0.9995
pr_auc: 0.9998

Files created:
1. ..\outputs\day9_training_history.csv
2. ..\outputs\day9_confusion_matrix.csv
3. ..\outputs\edge_gcn_model.pt
